# RLHF: 인간 피드백 기반 강화학습 - 실습 코드 1: RLHF Reward Model 학습 (TRL)

- Tutorial ID: `expand-rlhf`
- Tutorial: RLHF: 인간 피드백 기반 강화학습
- Section ID: `expand-rlhf-code-1`
- Section: 실습 코드 1: RLHF Reward Model 학습 (TRL)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: RLHF Reward Model 학습 (TRL)
#
# 이 코드는 "정답을 한 번 실행"하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 선호쌍 chosen/rejected가 loss와 policy update 신호로 바뀌는 흐름 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "shape 변화"와 "정보가 이동하는 방향"을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

## 0. 들어가기 전에 — RLHF와 Reward Model이 뭔가요?

### 이 노트북에서 배우는 순서

0. RLHF와 Reward Model 개념 이해
1. 실습 환경 준비 (라이브러리 설치)
2. 라이브러리 불러오기
3. 베이스 모델 선택 + `num_labels=1`의 의미 + 토크나이저/모델 구조 살펴보기
4. 선호도(preference) 데이터셋 만들기
5. 학습 설정(`RewardConfig`) 이해하기
6. Bradley-Terry Loss로 "학습 원리" 직접 확인하기
7. `RewardTrainer`로 실제 학습하기
8. 학습된 모델 테스트하기
9. 정리 및 다음 단계

---

ChatGPT 같은 모델이 "더 도움이 되고, 더 예의 바르고, 더 안전한" 답변을 하도록 만드는 대표적인 방법이 **RLHF (Reinforcement Learning from Human Feedback, 인간 피드백 기반 강화학습)** 입니다. RLHF는 보통 아래 3단계로 진행됩니다.

1. **SFT (Supervised Fine-Tuning)** — 사람이 직접 작성한 좋은 답변 예시로 모델을 1차로 지도학습시킵니다.
2. **Reward Model (RM) 학습** — "이 답변이 저 답변보다 낫다"는 사람의 선호도(preference) 데이터를 이용해, 답변에 점수를 매기는 채점관(scorer) 모델을 학습시킵니다. ← **이번 실습에서 만들 것**
3. **RL 최적화 (예: PPO)** — 2단계에서 만든 채점관의 점수를 "보상(reward)"으로 삼아, 원래 모델(정책, policy)이 더 높은 점수를 받는 방향으로 강화학습을 진행합니다.

이번 실습 코드는 이 중 **2단계, Reward Model 학습**만 다룹니다. 3단계(PPO)는 다음 실습에서 다룹니다.

### Reward Model을 한마디로 비유하면?

Reward Model은 "글쓰기 대회 심사위원"과 비슷합니다. 심사위원에게 채점 기준을 코드로 짜서 알려주는 대신, "이 글이 저 글보다 낫다"는 **비교 예시**를 많이 보여주면, 심사위원은 점점 사람의 취향을 흉내 내서 새로운 글에도 점수를 매길 수 있게 됩니다.

- 사람이 직접 "좋은 답변이란 이런 것이다"라는 절대적인 채점 기준을 코드로 짜기는 매우 어렵지만,
- "A가 B보다 낫다"는 **상대 비교**는 사람이 비교적 쉽게 판단할 수 있습니다.

그래서 RLHF는 절대 점수보다 **쌍대비교(pairwise comparison)** 데이터를 사용합니다. 이 실습에서 다룰 데이터 한 묶음(pair)은 다음과 같은 구조입니다.

| 필드 | 의미 | 예시 |
|---|---|---|
| `prompt` | 질문/지시문 | `"Explain AI"` |
| `chosen` | 더 선호되는(더 나은) 답변 | `"AI is the simulation of human intelligence..."` |
| `rejected` | 덜 선호되는(더 나쁜) 답변 | `"AI is robots and stuff"` |

Reward Model은 이런 쌍(pair)을 많이 보면서 **"`chosen`에는 높은 점수, `rejected`에는 낮은 점수"**를 주도록 학습됩니다. 아래에서 하나씩 직접 만들어 보겠습니다.

## 1. 실습 환경 준비

이 노트북은 아래 라이브러리를 사용합니다. Google Colab이라면 GPU가 없어도(CPU) 끝까지 실행은 가능하지만 시간이 걸릴 수 있으니, 가능하면 `런타임 > 런타임 유형 변경 > GPU`로 설정해두는 것을 권장합니다.

- **torch**: 딥러닝 연산(텐서 계산, 역전파)을 담당하는 기본 프레임워크
- **transformers**: 사전학습된 언어모델과 토크나이저를 손쉽게 불러오는 Hugging Face 라이브러리
- **trl** (Transformer Reinforcement Learning): SFT / Reward Model / PPO / DPO 등 RLHF 파이프라인 전용 학습 도구를 제공하는 라이브러리
- **datasets**: 학습 데이터를 다루기 쉬운 표 형태(`Dataset`)로 관리해 주는 라이브러리
- **accelerate**: GPU 등 학습 장치를 다루는 걸 도와주는 보조 라이브러리 (`trl`이 내부적으로 사용)

아래 셀은 처음 한 번만 실행하면 됩니다. 이미 설치되어 있다면 곧바로 스킵되니 여러 번 실행해도 안전합니다.

In [ ]:
# 처음 실행할 때만 필요합니다. (이미 설치돼 있으면 몇 초 만에 스킵됩니다)
# -q 옵션은 설치 로그를 간략하게 보여줍니다 (quiet).
!pip install -q transformers trl datasets accelerate

## 2. 필요한 도구(라이브러리) 불러오기

본격적인 코드에 들어가기 전에, 각 도구가 무엇을 위한 것인지 먼저 짚고 넘어갑니다.

- `AutoModelForSequenceClassification`: 언어모델 위에 "숫자 하나(점수)를 출력하는 헤드(head)"를 얹어주는 클래스입니다. 원래는 감정분석(긍정/부정 분류) 등에 쓰이던 클래스인데, Reward Model도 결국 "텍스트를 입력받아 숫자 하나를 출력"하는 구조라 이 클래스를 그대로 재활용합니다. (이유는 바로 아래 3번에서 자세히 설명합니다.)
- `AutoTokenizer`: 사람이 읽는 문장을 모델이 이해할 수 있는 숫자(token id) 배열로 바꿔주는 도구입니다.
- `RewardTrainer`, `RewardConfig`: `trl` 라이브러리가 제공하는, Reward Model 학습 전용 트레이너와 설정값 클래스입니다. `chosen`/`rejected` 쌍을 받아서 내부적으로 "토큰화 → 점수 계산 → loss 계산 → 역전파"까지 전부 알아서 처리해 줍니다.
- `Dataset`: 리스트나 딕셔너리 형태의 데이터를 Hugging Face 생태계에서 다루기 쉬운 표(테이블) 형태 객체로 바꿔주는 클래스입니다.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from trl import RewardTrainer, RewardConfig
from datasets import Dataset

# 실행 환경에 GPU가 있는지 확인합니다.
# GPU가 있으면 "cuda", 없으면 "cpu"로 표시됩니다. (Colab 무료 GPU는 보통 "cuda"로 뜹니다)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 중인 장치(device): {device}")

## 3. Reward Model의 베이스 모델 선택하기

원본 코드는 `meta-llama/Llama-2-7b-hf` (파라미터 약 70억 개)를 사용했습니다. 실제 서비스에서는 이런 대형 모델을 쓰지만, 이 실습에서는 두 가지 이유로 훨씬 작은 모델을 사용합니다.

1. **접근 권한 문제**: Llama-2는 Meta의 라이선스 동의 + Hugging Face 계정 승인이 필요한 "게이트(gated)" 모델이라, 승인 없이는 바로 다운로드할 수 없습니다.
2. **리소스 문제**: 70억 개 파라미터 모델은 학습에 최소 수십 GB의 GPU 메모리가 필요해서, 일반적인 실습 환경(Colab 무료 GPU 등)에서는 돌리기 어렵습니다.

그래서 이 실습에서는 **`gpt2`** (파라미터 약 1.24억 개, Llama-2의 약 1/60 크기)를 사용합니다. `gpt2`는 누구나 바로 내려받을 수 있고, CPU에서도 (조금 느리더라도) 실행 가능할 만큼 작습니다. **작동 원리는 완전히 동일**하므로, 여기서 익힌 코드는 나중에 그대로 큰 모델에도 적용할 수 있습니다.

### `num_labels=1`은 무슨 뜻인가요?

`AutoModelForSequenceClassification`은 원래 "분류(classification)"를 위한 클래스입니다. 예를 들어 감정분석이라면:

- `num_labels=2` → [부정, 긍정] 중 하나를 고르는 2지선다 분류
- `num_labels=5` → 별점 1~5점 중 하나를 고르는 5지선다 분류

반면 Reward Model이 원하는 출력은 "이 답변이 얼마나 좋은가"를 나타내는 **연속적인 숫자 하나**(예: -2.3, 0.8, 5.1 ...)입니다. 이건 분류가 아니라 회귀(regression)에 가깝죠. 그래서 `num_labels=1`로 설정하면, 모델 마지막에 "클래스 하나짜리 분류 헤드"가 붙는데, 이는 결과적으로 **숫자 하나를 출력하는 회귀 헤드와 동일하게 동작**합니다. 이 숫자 하나가 바로 우리가 원하는 **보상 점수(reward score)**입니다.

In [ ]:
# 이 실습에서 사용할 베이스 모델 이름입니다.
# 실제 서비스에서는 더 큰 모델(예: Llama-2-7b, Qwen 등)을 쓰지만
# 실습에서는 가볍게 돌려볼 수 있는 소형 모델 gpt2를 사용합니다.
model_name = "gpt2"

# 1) 토크나이저 불러오기: 문장 <-> 숫자(token id) 변환을 담당합니다.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# gpt2는 원래 '패딩 토큰(pad token)'이 정의되어 있지 않습니다.
# 패딩 토큰은 길이가 서로 다른 문장들을 하나의 batch로 묶을 때, 짧은 문장 뒤에 채워 넣는 '빈 자리' 토큰입니다.
#   예) "안녕"   -> [10, 20, <pad>, <pad>]  (짧은 문장 뒤를 <pad>로 채워 길이를 맞춤)
#       "반가워요" -> [11, 21, 31, 41]
# gpt2는 pad_token이 없으므로, "문장이 끝났다"는 걸 나타내는 eos_token(문장 종료 토큰)을
# 패딩 토큰으로 대신 사용하도록 설정합니다. (Reward Model 학습에서 흔히 쓰는 방법입니다)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2) 모델 불러오기: num_labels=1 -> "숫자 하나(점수)"를 출력하는 헤드를 gpt2 위에 얹습니다.
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

# 모델에게도 "패딩 토큰이 무엇인지" 알려줘야 합니다.
# (그래야 모델이 <pad> 부분은 무시하고, 진짜 문장이 끝나는 위치에서 점수를 뽑아냅니다)
model.config.pad_token_id = tokenizer.pad_token_id

model.to(device)

print(f"모델 이름: {model_name}")
print(f"전체 파라미터 개수: {model.num_parameters():,}개")
print(f"패딩 토큰: '{tokenizer.pad_token}' (id: {tokenizer.pad_token_id})")

## 3-1. (실험) 토크나이저가 문장을 어떻게 숫자로 바꾸는지 확인하기

모델은 문자를 그대로 이해하지 못하고, 반드시 숫자로 변환된 입력을 받습니다. 본격적으로 데이터셋을 만들기 전에, 짧은 문장 하나로 토크나이저가 실제로 어떤 값을 만들어내는지 눈으로 확인해봅시다.

In [ ]:
sample_text = "AI is the simulation of human intelligence."

# return_tensors="pt" -> 결과를 PyTorch 텐서(tensor) 형태로 반환합니다.
encoded = tokenizer(sample_text, return_tensors="pt")

print("원본 문장:", sample_text)
print("input_ids (토큰 id 배열):", encoded["input_ids"])
print("attention_mask (실제 토큰=1, 패딩=0):", encoded["attention_mask"])
print("input_ids shape:", encoded["input_ids"].shape)  # (배치 크기, 시퀀스 길이)

# 반대로, 토큰 id를 다시 사람이 읽는 조각(token)으로 되돌려볼 수도 있습니다.
decoded_tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
print("\n토큰 단위로 쪼갠 결과:", decoded_tokens)

## 3-2. (선택) 모델 구조 살짝 엿보기

gpt2는 내부적으로 여러 층(layer)의 self-attention 블록으로 이루어져 있습니다. 각 층에서 Q(Query)/K(Key)/V(Value) 벡터가 만들어지고, 이를 이용해 attention score를 계산합니다. (Q/K/V와 attention의 자세한 동작 원리는 이 시리즈의 다른 실습 — Transformer/Attention 편 — 에서 직접 다룹니다.) 여기서는 우리가 사용하는 gpt2가 몇 개의 층과 헤드로 이루어져 있는지만 간단히 확인해봅니다.

In [ ]:
print(f"레이어(층) 수 (n_layer): {model.config.n_layer}")
print(f"임베딩 차원 (n_embd, hidden size): {model.config.n_embd}")
print(f"어텐션 헤드 수 (n_head): {model.config.n_head}")
print(f"헤드 하나당 차원 (n_embd / n_head): {model.config.n_embd // model.config.n_head}")
print(f"\n분류/회귀 헤드 출력 차원 (num_labels): {model.config.num_labels}")  # 우리가 1로 설정한 값

## 4. 선호도(Preference) 데이터셋 만들기

Reward Model 학습에는 정답 레이블이 아니라, **"이 답변이 저 답변보다 낫다"는 비교 쌍(pair)**이 필요합니다. 각 데이터는 아래 3개 필드로 구성됩니다.

- `prompt`: 사용자의 질문/지시문
- `chosen`: 더 선호되는 답변 (사람이 "이게 더 낫다"고 고른 답변)
- `rejected`: 덜 선호되는 답변

원본 코드에는 예시가 딱 1개뿐이었고 `# ... 더 많은 데이터`라는 주석만 있었습니다. 실제 학습에는 최소 수백~수천 개가 필요하지만, 이 실습에서는 개념 이해를 위해 **10개의 예시**를 직접 만들어보겠습니다. (실전에서는 Anthropic의 `hh-rlhf` 같은 공개 데이터셋을 사용합니다. 맨 아래 "다음 단계"에서 안내합니다.)

아래 데이터에서 `chosen`과 `rejected`를 가르는 기준은 대략 이렇습니다.

- 더 구체적이고 정확한 설명 vs 대충 얼버무린 답변
- 성의 있고 도움이 되는 답변 vs 불친절하거나 무성의한 답변

In [ ]:
# prompt : 질문/지시문
# chosen : 더 선호되는(더 나은) 답변
# rejected : 덜 선호되는(더 나쁜) 답변
data = [
    {
        "prompt": "Explain AI",
        "chosen": "AI is the simulation of human intelligence processes by computer systems, including learning, reasoning, and self-correction.",
        "rejected": "AI is robots and stuff.",
    },
    {
        "prompt": "What is machine learning?",
        "chosen": "Machine learning is a subset of AI where systems learn patterns from data instead of being explicitly programmed with rules.",
        "rejected": "It's when computers learn things I guess.",
    },
    {
        "prompt": "How do I lose weight safely?",
        "chosen": "Focus on a balanced diet, regular exercise, and gradual, sustainable changes. Consult a doctor before starting any major diet or exercise plan.",
        "rejected": "Just stop eating for a few days, it works fast.",
    },
    {
        "prompt": "Explain what a neural network is.",
        "chosen": "A neural network is a computing system inspired by the brain, made of layers of connected nodes that learn to map inputs to outputs through training.",
        "rejected": "It's like a brain thing computers have now.",
    },
    {
        "prompt": "Give me tips for a job interview.",
        "chosen": "Research the company beforehand, prepare specific examples of your past work, and practice clear, concise answers to common questions.",
        "rejected": "Just wing it, interviews don't really matter that much.",
    },
    {
        "prompt": "What should I do if my code has a bug?",
        "chosen": "Reproduce the bug consistently, check the error message carefully, isolate the failing part with small tests, and use a debugger or print statements to trace the issue.",
        "rejected": "idk just delete stuff until it works",
    },
    {
        "prompt": "How does the internet work?",
        "chosen": "The internet is a global network of computers that communicate using standardized protocols like TCP/IP, allowing data to be broken into packets, routed, and reassembled.",
        "rejected": "It's just magic wires that connect everything.",
    },
    {
        "prompt": "What is a healthy breakfast?",
        "chosen": "A healthy breakfast typically includes protein, whole grains, and fruit or vegetables, such as oatmeal with berries and nuts, or eggs with whole-grain toast.",
        "rejected": "Skip it, breakfast doesn't matter.",
    },
    {
        "prompt": "How can I stay safe online?",
        "chosen": "Use strong, unique passwords, enable two-factor authentication, avoid clicking suspicious links, and keep your software updated.",
        "rejected": "Just click whatever, nothing bad ever happens.",
    },
    {
        "prompt": "Explain gravity in simple terms.",
        "chosen": "Gravity is the force that pulls objects with mass toward each other; it's why things fall to the ground and why planets orbit the sun.",
        "rejected": "Gravity is just stuff falling down, don't overthink it.",
    },
]

print(f"데이터 개수: {len(data)}개")
print("\n첫 번째 데이터 예시:")
print(data[0])

In [ ]:
# 파이썬 리스트(list of dict)를 Hugging Face의 Dataset 객체로 변환합니다.
# Dataset 객체로 바꾸면 RewardTrainer가 바로 이해할 수 있는 표(테이블) 형태가 됩니다.
dataset = Dataset.from_list(data)

print(dataset)
print("\n첫 번째 데이터 샘플 (Dataset에서 꺼내보기):")
print(dataset[0])

## 5. 학습 설정값 정하기 — `RewardConfig`

`RewardConfig`는 Hugging Face의 `TrainingArguments`를 상속받아, "학습을 어떻게 진행할지"에 대한 설정값들을 모아놓은 객체입니다. 이 실습에서 사용하는 주요 옵션은 다음과 같습니다.

| 옵션 | 의미 | 이 실습 값 |
|---|---|---|
| `output_dir` | 학습 결과(체크포인트, 로그)를 저장할 폴더 경로 | `"./reward_model"` |
| `per_device_train_batch_size` | 한 번의 학습 스텝(step)에서 GPU/CPU 1개가 동시에 처리하는 데이터 쌍(pair) 개수 | `2` |
| `learning_rate` | 한 스텝마다 가중치를 얼마나 크게 업데이트할지 정하는 값 | `1e-5` |
| `num_train_epochs` | 전체 데이터셋을 몇 번 반복해서 학습할지 | `3` |
| `logging_steps` | 몇 스텝마다 loss 등 학습 로그를 출력할지 | `1` |
| `report_to` | 학습 로그를 어디로 보낼지 (wandb 등 외부 연동 없이 콘솔 출력만 하려면 `"none"`) | `"none"` |
| `max_length` | 토큰 기준 최대 길이. 이보다 길면 해당 데이터는 학습에서 제외됨 | 지정 안 함 (기본값 1024, 이 실습 문장들은 짧아서 충분) |

> 참고: 예제 데이터가 10개뿐이라 `per_device_train_batch_size`를 원본 코드의 4보다 작은 2로 낮췄습니다. (배치 크기가 전체 데이터 개수보다 크면 비효율적입니다) 실제 프로젝트에서는 데이터 양과 GPU 메모리에 맞춰 배치 크기를 조정합니다.

In [ ]:
training_args = RewardConfig(
    output_dir="./reward_model",       # 학습 결과가 저장될 폴더
    per_device_train_batch_size=2,     # 한 스텝에 2개 쌍(pair)씩 학습 (예제 데이터가 적어서 작게 설정)
    learning_rate=1e-5,                # 가중치 업데이트 보폭 (너무 크면 학습이 불안정해지고, 너무 작으면 학습이 느립니다)
    num_train_epochs=3,                # 데이터 10개를 3번 반복 학습 (데이터가 적으므로 여러 epoch가 필요)
    logging_steps=1,                   # 매 스텝마다 로그 출력 (데이터가 적어 스텝 수도 적으므로 자주 확인)
    report_to="none",                  # wandb 등 외부 로깅 툴 연동 없이, 콘솔에만 로그 출력
)

print("학습 결과 저장 경로:", training_args.output_dir)
print("배치 크기:", training_args.per_device_train_batch_size)
print("학습률(learning rate):", training_args.learning_rate)
print("총 epoch 수:", training_args.num_train_epochs)

## 6. Reward Model은 어떻게 학습되나요? — Loss 함수 살짝 들여다보기

`RewardTrainer`를 실행하면 내부적으로 매 스텝마다 아래 과정이 반복됩니다.

1. 모델이 `prompt + chosen`을 읽고 점수 하나를 출력합니다 → `reward_chosen`
2. 모델이 `prompt + rejected`를 읽고 점수 하나를 출력합니다 → `reward_rejected`
3. 두 점수의 차이(`reward_chosen - reward_rejected`)를 가지고 **loss(손실값)**를 계산합니다.
4. loss가 작아지는 방향으로 모델 가중치를 업데이트합니다 (역전파).

이때 사용하는 손실 함수를 **Bradley-Terry loss**라고 부르며, 수식은 다음과 같습니다.

```
loss = -log( sigmoid(reward_chosen - reward_rejected) )
```

말로 풀면: **"chosen 점수가 rejected 점수보다 많이 높을수록 loss는 0에 가까워지고, 반대로 rejected 점수가 더 높으면(모델이 틀렸으면) loss가 커진다"**는 뜻입니다. 실제 코드/모델 없이, 숫자만 가정해서 아래에서 직접 확인해봅시다.

In [ ]:
import torch.nn.functional as F
# torch.nn.functional(F)은 활성화 함수, 손실 함수 등 다양한 신경망 연산을 모아둔 모듈입니다.

def bradley_terry_loss(reward_chosen, reward_rejected):
    # chosen 점수와 rejected 점수를 받아 Bradley-Terry loss를 계산합니다.
    diff = reward_chosen - reward_rejected
    # -log(sigmoid(x))는 torch.nn.functional의 logsigmoid를 이용해 수치적으로 안전하게 계산할 수 있습니다.
    loss = -F.logsigmoid(diff)
    return loss.item()

# 케이스별로 chosen/rejected 점수를 가정해서 loss가 어떻게 달라지는지 확인합니다.
cases = [
    ("모델이 정확히 맞춤 (chosen이 훨씬 높음)", torch.tensor(5.0), torch.tensor(-3.0)),
    ("모델이 얼추 맞춤 (chosen이 약간 높음)",   torch.tensor(0.5), torch.tensor(0.1)),
    ("모델이 확신이 없음 (거의 동점)",          torch.tensor(0.0), torch.tensor(0.0)),
    ("모델이 완전히 틀림 (rejected가 더 높음)", torch.tensor(-2.0), torch.tensor(3.0)),
]

print(f"{'상황':38s} | reward_chosen | reward_rejected | loss")
print("-" * 90)
for name, r_chosen, r_rejected in cases:
    loss = bradley_terry_loss(r_chosen, r_rejected)
    print(f"{name:38s} | {r_chosen.item():>13.1f} | {r_rejected.item():>16.1f} | {loss:.4f}")

실행해보면 다음 패턴을 확인할 수 있습니다.

- **점수 차이가 크고 방향이 맞을수록** (1번 케이스) loss는 0에 가까워집니다.
- **점수가 거의 같으면** (3번 케이스) loss는 약 `0.693`(= ln 2)이 됩니다. 이는 "동전 던지기"처럼 완전히 반반으로 헷갈리는 상태를 의미합니다.
- **방향이 틀리면** (4번 케이스, rejected 점수가 더 높음) loss가 크게 튀어 오릅니다.

학습 과정에서 이 loss를 줄이는 방향으로 가중치가 업데이트되면서, 모델은 점점 `chosen`에 더 높은 점수를 주도록 바뀝니다. `RewardTrainer`의 학습 로그에 찍히는 `loss`가 바로 이 값이고, `accuracy`는 "매 배치에서 chosen 점수가 rejected 점수보다 높았던 비율"입니다. 학습이 잘 되고 있다면 **loss는 점점 낮아지고, accuracy는 점점 1.0(100%)에 가까워집니다.**

## 7. RewardTrainer 만들고 학습 시작하기

이제 지금까지 준비한 것들을 모두 모아 `RewardTrainer`를 만듭니다.

- `model`: 3번에서 불러온 gpt2 기반 Reward Model
- `args`: 5번에서 만든 학습 설정 (`RewardConfig`)
- `train_dataset`: 4번에서 만든 선호도 데이터셋
- `processing_class`: 토크나이저 (예전 버전 TRL/transformers 튜토리얼에서는 `tokenizer=`라는 이름의 인자를 쓰기도 하지만, 최신 버전에서는 `processing_class=`를 사용합니다)

`RewardTrainer`를 생성하는 순간, 내부적으로 `dataset`의 `prompt`+`chosen`, `prompt`+`rejected` 텍스트를 자동으로 토큰화합니다. 즉, 우리가 3-1에서 직접 해봤던 "문장 → input_ids 변환" 과정을 각 데이터마다 알아서 처리해주는 것입니다.

In [ ]:
trainer = RewardTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,  # 예전 코드에서는 tokenizer=tokenizer 로 쓰기도 합니다.
)

print("RewardTrainer가 준비되었습니다. 학습을 시작할 수 있습니다.")

아래 셀을 실행하면 실제 학습이 시작됩니다. 데이터가 10개뿐이라 CPU에서도 수 분 내로 끝납니다. 로그에 찍히는 `loss`는 점점 작아지고, `accuracy`(정확도, chosen에 더 높은 점수를 준 비율)는 점점 커지는지 확인해보세요.

In [ ]:
train_result = trainer.train()

print("\n학습 완료!")
print("최종 학습 결과:", train_result.metrics)

## 8. 학습된 Reward Model 테스트해보기

학습이 끝난 모델이 정말로 "더 나은 답변"에 더 높은 점수를 주는지 직접 확인해봅시다. 이번에는 **학습 데이터에 없던 새로운 문장**으로 테스트합니다. 아래 함수는 3-1에서 해봤던 토큰화 과정과 거의 동일하되, 토큰화한 결과를 모델에 직접 통과시켜 점수(logits)를 뽑아내는 과정이 추가되었습니다.

In [ ]:
def get_reward_score(prompt, response):
    # prompt + response를 모델에 넣어 보상 점수(숫자 하나)를 반환합니다.
    text = prompt + response
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():  # 테스트 단계이므로 gradient(기울기) 계산은 필요 없습니다.
        outputs = model(**inputs)

    # outputs.logits의 shape은 (배치 크기=1, num_labels=1) 입니다.
    # 우리가 원하는 건 그 안의 숫자 하나이므로 .item()으로 순수 숫자만 꺼냅니다.
    score = outputs.logits.item()
    return score


# 학습 데이터에 없던 새로운 예시로 테스트합니다.
test_prompt = "What is the best way to learn programming?"
test_good_response = "Start with the fundamentals, build small projects, and practice consistently while reading others' code."
test_bad_response = "idk just watch youtube videos i guess lol"

score_good = get_reward_score(test_prompt, test_good_response)
score_bad = get_reward_score(test_prompt, test_bad_response)

print(f"좋은 답변 점수 : {score_good:.4f}")
print(f"나쁜 답변 점수 : {score_bad:.4f}")

if score_good > score_bad:
    print("\n[OK] 모델이 더 나은 답변에 더 높은 점수를 줬습니다!")
else:
    print("\n[주의] 아직 두 점수가 뒤집혀 있습니다. 데이터가 10개뿐인 미니 실습이라 "
          "완벽하게 학습되지 않을 수 있습니다. epoch을 늘리거나 데이터를 더 추가해보세요.")

## 9. 정리 및 다음 단계

이번 실습에서 한 일을 정리하면 다음과 같습니다.

1. RLHF 파이프라인에서 Reward Model이 어떤 역할을 하는지 개념적으로 이해했습니다.
2. `chosen`/`rejected` 선호도 데이터가 어떤 구조인지 직접 만들어봤습니다.
3. `AutoModelForSequenceClassification(num_labels=1)`로 "점수 하나를 출력하는" 모델을 gpt2 위에 얹었습니다.
4. Bradley-Terry loss가 어떻게 "chosen에는 높은 점수, rejected에는 낮은 점수"를 유도하는지 숫자로 확인했습니다.
5. `RewardTrainer`로 실제 학습을 돌리고, 학습된 모델이 새로운 문장에 점수를 매기는 것까지 확인했습니다.

### 더 실험해보고 싶다면

- `num_train_epochs`를 5, 10으로 늘려보고 loss/accuracy가 어떻게 변하는지 관찰해보세요.
- `data` 리스트에 직접 예시를 5~10개 더 추가해보세요.
- `dataset`을 `dataset.train_test_split(test_size=0.2)`로 나눠서 `eval_dataset`과 `eval_strategy`를 추가하면, 학습에 쓰지 않은 데이터로 성능을 점검할 수 있습니다.
- 실전 데이터셋으로 연습하고 싶다면 Hugging Face Hub의 `Anthropic/hh-rlhf`, `trl-internal-testing/hh-rlhf-trl-style` 같은 공개 선호도 데이터셋을 `datasets.load_dataset()`으로 불러와 그대로 `dataset` 자리에 넣어볼 수 있습니다.
- `model_name`을 `"gpt2"`에서 `"gpt2-medium"` 등 더 큰 모델로 바꿔보고 결과가 달라지는지 비교해보세요.

### 다음 실습 코드에서는?

이번에 학습한 Reward Model은 "채점관"입니다. 다음 실습에서는 이 채점관이 매기는 점수를 실제 "보상(reward)" 신호로 사용해서, 원래 언어모델(정책, policy)을 PPO 같은 강화학습 알고리즘으로 업데이트하는 과정을 다룹니다.